<a href="https://colab.research.google.com/github/petrovortex/foundations_of_ml_course/blob/main/applied_ML_%5BCNN%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Провести анализ качества аппроксимации выборки EMNIST-letters моделью сверточной нейронной сети в зависимости от:

- размера ядра (можно ввести ограничение, что на каждом слое размер ядра одинаковый);
- числа слоев;
- от пулинга;
- добавления BatchNorm;
- параметра dropout.

## CNN & Trainer

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Optimizer
from torch.utils.tensorboard import SummaryWriter
from typing import Tuple, List, Dict
from tqdm.auto import tqdm
import copy

In [ ]:
class ConfigurableCNN(nn.Module):
    def __init__(
        self,
        input_channels: int = 1,
        num_classes: int = 26,
        num_layers: int = 2,
        kernel_size: int = 3,
        use_pooling: bool = True,
        use_batchnorm: bool = True,
        dropout_rate: float = 0.0
    ) -> None:
        super().__init__()

        layers = []
        current_channels = input_channels
        current_size = 28

        for i in range(num_layers):
            out_channels = 16 * (2 ** i)
            padding = kernel_size // 2

            layers.append(
                nn.Conv2d(
                    in_channels=current_channels,
                    out_channels=out_channels,
                    kernel_size=kernel_size,
                    padding=padding
                )
            )

            if use_batchnorm:
                layers.append(nn.BatchNorm2d(out_channels))

            layers.append(nn.ReLU())

            if use_pooling:
                layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
                current_size //= 2

            if dropout_rate > 0.0:
                layers.append(nn.Dropout2d(p=dropout_rate))

            current_channels = out_channels

        self.feature_extractor = nn.Sequential(*layers)

        flattened_size = current_channels * current_size * current_size

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flattened_size, num_classes)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.feature_extractor(x)
        logits = self.classifier(features)
        return logits

In [ ]:
class Trainer:
    def __init__(
        self,
        model: nn.Module,
        optimizer: Optimizer,
        criterion: nn.Module,
        train_loader: DataLoader,
        val_loader: DataLoader,
        device: torch.device,
        writer: SummaryWriter
    ) -> None:
        self.model = model.to(device)
        self.optimizer = optimizer
        self.criterion = criterion
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.writer = writer

    def _train_epoch(self, epoch_idx: int, num_epochs: int) -> Tuple[float, float]:
        self.model.train()
        total_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        progress_bar = tqdm(
            self.train_loader,
            desc=f"Train Epoch {epoch_idx}/{num_epochs}",
            leave=False
        )

        for inputs, targets in progress_bar:
            inputs = inputs.to(self.device)
            targets = targets.to(self.device)

            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs, 1)
            correct_predictions += (predicted == targets).sum().item()
            total_samples += targets.size(0)

            progress_bar.set_postfix({"loss": loss.item()})

        avg_loss = total_loss / total_samples
        accuracy = correct_predictions / total_samples
        return avg_loss, accuracy

    def _validate_epoch(self, epoch_idx: int, num_epochs: int) -> Tuple[float, float]:
        self.model.eval()
        total_loss = 0.0
        correct_predictions = 0
        total_samples = 0

        progress_bar = tqdm(
            self.val_loader,
            desc=f"Val Epoch {epoch_idx}/{num_epochs}",
            leave=False
        )

        with torch.no_grad():
            for inputs, targets in progress_bar:
                inputs = inputs.to(self.device)
                targets = targets.to(self.device)

                outputs = self.model(inputs)
                loss = self.criterion(outputs, targets)

                total_loss += loss.item() * inputs.size(0)
                _, predicted = torch.max(outputs, 1)
                correct_predictions += (predicted == targets).sum().item()
                total_samples += targets.size(0)

        avg_loss = total_loss / total_samples
        accuracy = correct_predictions / total_samples
        return avg_loss, accuracy

    def fit(self, num_epochs: int) -> List[Dict[str, float]]:
        metrics_history = []

        for epoch in range(1, num_epochs + 1):
            train_loss, train_acc = self._train_epoch(epoch, num_epochs)
            val_loss, val_acc = self._validate_epoch(epoch, num_epochs)

            self.writer.add_scalar("Loss/Train", train_loss, epoch)
            self.writer.add_scalar("Accuracy/Train", train_acc, epoch)
            self.writer.add_scalar("Loss/Validation", val_loss, epoch)
            self.writer.add_scalar("Accuracy/Validation", val_acc, epoch)

            metrics_history.append({
                "Loss/Train": train_loss,
                "Accuracy/Train": train_acc,
                "Loss/Validation": val_loss,
                "Accuracy/Validation": val_acc
            })

        return metrics_history

## Data loading & Experiments

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

In [ ]:
import torchvision.transforms as transforms
from torchvision.datasets import EMNIST
import torch.optim as optim

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

target_transform = lambda x: x - 1

train_dataset = EMNIST(
    root='./data',
    split='letters',
    train=True,
    download=True,
    transform=transform,
    target_transform=target_transform
)

val_dataset = EMNIST(
    root='./data',
    split='letters',
    train=False,
    download=True,
    transform=transform,
    target_transform=target_transform
)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=128, shuffle=False)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

baseline_config = {
    "num_layers": 2,
    "kernel_size": 3,
    "use_pooling": True,
    "use_batchnorm": True,
    "dropout_rate": 0.0
}

experiments = {
    "kernel_size": [3, 5, 7],
    "num_layers": [1, 2, 3],
    "use_pooling": [True, False],
    "use_batchnorm": [True, False],
    "dropout_rate": [0.0, 0.25, 0.5]
}

num_epochs = 5

In [ ]:
baseline_writer = SummaryWriter(log_dir="runs/baseline/run")
baseline_model = ConfigurableCNN(**baseline_config).to(device)
baseline_optimizer = torch.optim.Adam(baseline_model.parameters(), lr=1e-3)

print("Training baseline model...")
baseline_trainer = Trainer(
    model=baseline_model,
    optimizer=baseline_optimizer,
    criterion=criterion,
    train_loader=train_loader,
    val_loader=val_loader,
    device=device,
    writer=baseline_writer
)

baseline_history = baseline_trainer.fit(num_epochs)
baseline_writer.close()

for param_name, values in experiments.items():
    for value in values:
        run_name = f"runs/{param_name}/{value}"
        writer = SummaryWriter(log_dir=run_name)

        if value == baseline_config[param_name]:
            print(f"Skipping training for {param_name} = {value} (using baseline metrics)")
            for epoch_idx, metrics in enumerate(baseline_history, start=1):
                for metric_name, metric_value in metrics.items():
                    writer.add_scalar(metric_name, metric_value, epoch_idx)
            writer.close()
            continue

        print(f"Training configuration: {param_name} = {value}")

        current_config = copy.deepcopy(baseline_config)
        current_config[param_name] = value

        model = ConfigurableCNN(**current_config).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

        trainer = Trainer(
            model=model,
            optimizer=optimizer,
            criterion=criterion,
            train_loader=train_loader,
            val_loader=val_loader,
            device=device,
            writer=writer
        )

        trainer.fit(num_epochs)
        writer.close()

## DELETE RUNS

In [ ]:
!rm -rf runs